# Limpeza e normalização

Transforma o dataset bruto por setor (`setores_sao_paulo_2022.csv`, gerado por `comunidades_ibge.ipynb`)
em uma base pronta para o modelo:

1. separa identificação, rótulo e features;
2. tira do modelo os setores sem uso (represas, sem moradores, Censo em sigilo);
3. trata os vazios que sobram, cada um pelo seu motivo;
4. divide treino, validação e teste por distrito;
5. normaliza as features com parâmetros do treino, aplicados igualmente aos três conjuntos;
6. salva a base tratada, os setores excluídos e os parâmetros em `data/output/`.

Os números citados no texto são da execução para São Paulo.

In [1]:
import json
import math
from collections.abc import Callable
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Final

import numpy as np
import pandas as pd

ROOT: Final = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT: Final = ROOT / "data" / "output"
SOURCE: Final = OUT / "setores_sao_paulo_2022.csv"
DATASET_PATH: Final = OUT / f"{SOURCE.stem}_tratado.csv"
EXCLUDED_PATH: Final = OUT / f"{SOURCE.stem}_excluidos.csv"
REPORT_PATH: Final = OUT / f"{SOURCE.stem}_tratamento.json"
SEED: Final = 42

# Grupos de colunas

- **Identificação**: não entra no modelo. `code_tract` é a chave que liga o setor à imagem. Latitude e
  longitude servem para localizar a imagem; como feature, fariam o modelo decorar a geografia.
  `area_m2` já está contida nas densidades.
- **Rótulo**: `favela` é o alvo. `pct_favela` é de onde o alvo sai, então usá-la como feature seria
  vazamento. `code_favela` e `name_favela` só descrevem a favela.
- **Features**: 32 contínuas, agrupadas pela fonte (malha, agregados do Censo, entorno), mais duas
  binárias criadas aqui: `zona_urbana` e `entorno_ausente`.
- `split` (`train`, `validation`, `test`) é criado na seção do split.

In [2]:
KEY: Final = "code_tract"
GROUP: Final = "name_district"
CITY_DISTRICT: Final = "distrito_prefeitura"
ZONE: Final = "zone"
URBAN_ZONE: Final = "urbana"
TARGET: Final = "favela"
POPULATION: Final = "populacao"
SPLIT: Final = "split"
EXCLUSION_REASON: Final = "exclusion_reason"
URBAN_FLAG: Final = "zona_urbana"
MISSING_SURROUNDINGS_FLAG: Final = "entorno_ausente"

IDENTIFICATION: Final[tuple[str, ...]] = (
    KEY,
    "code_muni",
    "name_muni",
    GROUP,
    CITY_DISTRICT,
    ZONE,
    "area_m2",
    "latitude",
    "longitude",
)

LABEL: Final[tuple[str, ...]] = (
    TARGET,
    "pct_favela",
    "code_favela",
    "name_favela",
)

TRACT_MESH: Final[tuple[str, ...]] = (
    POPULATION,
    "moradores_por_domicilio",
    "densidade_hab_km2",
    "densidade_domicilios_km2",
)

CENSUS_AGGREGATES: Final[tuple[str, ...]] = (
    "pct_0a14",
    "pct_15a29",
    "pct_60mais",
    "razao_sexo",
    "pct_branca",
    "pct_preta",
    "pct_parda",
    "pct_preta_parda",
    "taxa_alfabetizacao_15mais",
    "renda_media_responsavel",
    "pct_casa",
    "pct_casa_vila_condominio",
    "pct_apartamento",
    "pct_cortico_degradado",
    "pct_dom_1_morador",
    "pct_dom_5mais_moradores",
    "pct_agua_rede_geral",
    "pct_esgoto_adequado",
    "pct_lixo_coletado",
    "pct_banheiro_exclusivo",
)

SURROUNDINGS: Final[tuple[str, ...]] = (
    "pct_via_pavimentada",
    "pct_via_so_pedestre_moto",
    "pct_iluminacao_publica",
    "pct_bueiro",
    "pct_calcada",
    "pct_rampa_cadeirante",
    "pct_ponto_onibus",
    "pct_arborizacao",
)

CONTINUOUS: Final[tuple[str, ...]] = TRACT_MESH + CENSUS_AGGREGATES + SURROUNDINGS

LOG_TRANSFORMED: Final[frozenset[str]] = frozenset(
    {
        "densidade_hab_km2",
        "densidade_domicilios_km2",
        "razao_sexo",
        "renda_media_responsavel",
    }
)

BINARY: Final[tuple[str, ...]] = (URBAN_FLAG, MISSING_SURROUNDINGS_FLAG)

FEATURES: Final[tuple[str, ...]] = CONTINUOUS + BINARY

# Leitura

`code_tract` tem 15 dígitos e precisa ser lido como texto: como número, deixaria de bater com o nome das
imagens. `code_favela` também vira texto, porque com vazios o pandas leria como float e escreveria
`3.55e+10`. A leitura falha se a chave se repetir ou não tiver 15 dígitos.

In [3]:
_KEY_PATTERN: Final = r"\d{15}"
_TEXT_COLUMNS: Final[dict[str, type[str]]] = {
    KEY: str,
    "code_muni": str,
    "code_favela": str,
}


def _validate_key(frame: pd.DataFrame) -> None:
    keys = frame[KEY]
    if not keys.is_unique:
        raise ValueError(f"{KEY} has duplicated values")
    if not keys.str.fullmatch(_KEY_PATTERN).all():
        raise ValueError(f"{KEY} values must have 15 digits")


def read_sectors(source: Path) -> pd.DataFrame:
    sectors = pd.read_csv(source, dtype=_TEXT_COLUMNS)
    _validate_key(sectors)
    return sectors


sectors = read_sectors(SOURCE)
print(f"sectors: {len(sectors)} | columns: {sectors.shape[1]}")

sectors: 27301 | columns: 45


# Diagnóstico dos vazios

Colunas com a mesma contagem de vazios costumam ter a mesma causa: 622 nas features do Censo, 1.019 no
entorno, 10 em `zone`. As seções seguintes tratam cada causa separadamente.

In [4]:
def missing_counts(frame: pd.DataFrame) -> pd.Series:
    counts = frame.isna().sum()
    return counts[counts > 0]


missing_counts(sectors)

distrito_prefeitura              2
zone                            10
code_favela                  23426
name_favela                  23426
pct_0a14                       622
pct_15a29                      622
pct_60mais                     622
razao_sexo                     623
pct_branca                     622
pct_preta                      622
pct_parda                      622
pct_preta_parda                622
taxa_alfabetizacao_15mais      636
renda_media_responsavel        629
pct_casa                       622
pct_casa_vila_condominio       622
pct_apartamento                622
pct_cortico_degradado          622
pct_dom_1_morador              622
pct_dom_5mais_moradores        622
pct_agua_rede_geral            622
pct_esgoto_adequado            622
pct_lixo_coletado              622
pct_banheiro_exclusivo         622
pct_via_pavimentada           1019
pct_via_so_pedestre_moto      1019
pct_iluminacao_publica        1019
pct_bueiro                    1019
pct_calcada         

# Setores sem uso para o modelo

| `exclusion_reason` | regra | por quê |
|---|---|---|
| `water_body` | `zone` vazia | represas (CD_SIT = 9): sem moradores nem domicílios |
| `no_residents` | `populacao == 0` | as proporções sociais não têm denominador. Nesses setores `moradores_por_domicilio` vem como 0, um zero que significa "não se aplica" |
| `census_suppressed` | os 20 indicadores dos agregados vazios | o setor tem moradores (mediana de 8), mas o IBGE pôs todos os agregados em sigilo (`"X"`). Imputar os 20 seria inventar o perfil inteiro do setor |

O motivo é a primeira regra que vale para o setor (uma represa também tem `populacao == 0`). Os setores
não são apagados: vão para `_excluidos.csv` e continuam disponíveis para a etapa das imagens.

In [5]:
WATER_BODY: Final = "water_body"
NO_RESIDENTS: Final = "no_residents"
CENSUS_SUPPRESSED: Final = "census_suppressed"


@dataclass(frozen=True)
class SectorPartition:
    usable: pd.DataFrame
    excluded: pd.DataFrame


def _is_water_body(frame: pd.DataFrame) -> pd.Series:
    return frame[ZONE].isna()


def _has_no_residents(frame: pd.DataFrame) -> pd.Series:
    return frame[POPULATION].eq(0)


def _has_suppressed_census(frame: pd.DataFrame) -> pd.Series:
    return frame[list(CENSUS_AGGREGATES)].isna().all(axis=1)


_EXCLUSION_RULES: Final[dict[str, Callable[[pd.DataFrame], pd.Series]]] = {
    WATER_BODY: _is_water_body,
    NO_RESIDENTS: _has_no_residents,
    CENSUS_SUPPRESSED: _has_suppressed_census,
}


def _exclusion_reasons(frame: pd.DataFrame) -> pd.Series:
    conditions = [rule(frame) for rule in _EXCLUSION_RULES.values()]
    reasons = np.select(conditions, list(_EXCLUSION_RULES), default="")
    return pd.Series(reasons, index=frame.index, name=EXCLUSION_REASON)


def separate_unusable_sectors(frame: pd.DataFrame) -> SectorPartition:
    reasons = _exclusion_reasons(frame)
    excluded = reasons.ne("")
    return SectorPartition(
        usable=frame.loc[~excluded],
        excluded=frame.loc[excluded].assign(**{EXCLUSION_REASON: reasons[excluded]}),
    )


partition = separate_unusable_sectors(sectors)
print(
    f"usable: {len(partition.usable)} | excluded: {len(partition.excluded)} | "
    f"favela sectors excluded: {int(partition.excluded[TARGET].sum())} of {int(sectors[TARGET].sum())}"
)
partition.excluded[EXCLUSION_REASON].value_counts()

usable: 26679 | excluded: 622 | favela sectors excluded: 17 of 3875


exclusion_reason
no_residents         402
census_suppressed    210
water_body            10
Name: count, dtype: int64

# Vazios que permanecem

Depois da exclusão sobram estes vazios:

In [6]:
missing_counts(partition.usable)

distrito_prefeitura              2
code_favela                  22821
name_favela                  22821
razao_sexo                       1
taxa_alfabetizacao_15mais       14
renda_media_responsavel          7
pct_via_pavimentada            397
pct_via_so_pedestre_moto       397
pct_iluminacao_publica         397
pct_bueiro                     397
pct_calcada                    397
pct_rampa_cadeirante           397
pct_ponto_onibus               397
pct_arborizacao                397
dtype: int64

| caso | setores | tratamento | justificativa |
|---|---|---|---|
| entorno não pesquisado (8 colunas) | 397 (146 rurais, 251 urbanos) | mediana do treino + flag `entorno_ausente` | o IBGE não pesquisou o entorno em todos os setores (só 21 dos 167 rurais têm). A ausência diz algo sobre o setor, e a flag deixa o modelo separar valor medido de valor imputado |
| sigilo pontual: renda (7), alfabetização (14), razão de sexo (1, setor sem mulheres) | 22 | mediana do treino, sem flag | casos raros (< 0,1%) em setores pequenos. Uma flag quase constante não ensinaria nada ao modelo |
| `distrito_prefeitura` | 2 | distrito da Prefeitura mais frequente entre os setores do mesmo distrito IBGE (Pedreira → `PEDREIRA`) | as duas fontes têm os mesmos 96 distritos, e em cada distrito IBGE pelo menos 97% dos setores caem no mesmo distrito da Prefeitura |
| `code_favela`, `name_favela` | setores fora de favela | mantidos vazios | "não se aplica": descrevem o rótulo, não são features |

A mediana depende do treino, então a imputação acontece junto com a normalização, mais abaixo. Aqui ficam
o distrito e as flags, que não dependem do split. `zona_urbana` leva para as features a informação de
`zone` (urbana ou rural), que continua na identificação como texto.

Limitação herdada da coleta: `proporcao()` troca numerador sigiloso por 0, então parte dos zeros nas
proporções são vazios disfarçados. Não dá para recuperá-los a partir do CSV. Corrigir isso exige mudar a
coleta.

In [7]:
def _most_frequent(values: pd.Series) -> object:
    modes = values.mode()
    return pd.NA if modes.empty else modes.iat[0]


def _dominant_city_district(frame: pd.DataFrame) -> pd.Series:
    return frame.groupby(GROUP)[CITY_DISTRICT].agg(_most_frequent)


def fill_city_district(frame: pd.DataFrame) -> pd.DataFrame:
    dominant = frame[GROUP].map(_dominant_city_district(frame))
    filled = frame[CITY_DISTRICT].fillna(dominant)
    return frame.assign(**{CITY_DISTRICT: filled})


def _urban_flag(frame: pd.DataFrame) -> pd.Series:
    return frame[ZONE].eq(URBAN_ZONE).astype(int)


def _missing_surroundings_flag(frame: pd.DataFrame) -> pd.Series:
    return frame[list(SURROUNDINGS)].isna().all(axis=1).astype(int)


def add_binary_features(frame: pd.DataFrame) -> pd.DataFrame:
    return frame.assign(
        **{
            URBAN_FLAG: _urban_flag(frame),
            MISSING_SURROUNDINGS_FLAG: _missing_surroundings_flag(frame),
        }
    )


usable = add_binary_features(fill_city_district(partition.usable))
pd.crosstab(usable[ZONE], usable[MISSING_SURROUNDINGS_FLAG])

entorno_ausente,0,1
zone,,
rural,21,146
urbana,26261,251


# Split por distrito

Setores vizinhos são parecidos e as imagens de setores vizinhos se sobrepõem, então sortear setor a setor
vazaria informação entre treino e teste. Por isso o split é por distrito IBGE: os distritos são ordenados
pela taxa de favela e, em cada bloco de 10, sorteia-se 1 para teste e 1 para validação (`SEED`). Assim
cada conjunto recebe distritos com pouca e com muita favela, e a taxa fica parecida nos três.

In [8]:
TRAIN: Final = "train"
VALIDATION: Final = "validation"
TEST: Final = "test"
SPLITS: Final[tuple[str, ...]] = (TRAIN, VALIDATION, TEST)

STRATUM_SIZE: Final = 10
_TEST_SLOT: Final = 0
_VALIDATION_SLOT: Final = 1


def _districts_by_favela_rate(frame: pd.DataFrame) -> pd.Index:
    rates = frame.groupby(GROUP)[TARGET].mean()
    return rates.sort_values(kind="stable").index


def _shuffled_slots(n_districts: int, seed: int) -> np.ndarray:
    generator = np.random.default_rng(seed)
    n_strata = math.ceil(n_districts / STRATUM_SIZE)
    slots = [generator.permutation(STRATUM_SIZE) for _ in range(n_strata)]
    return np.concatenate(slots)[:n_districts]


def _split_of_slot(slot: int) -> str:
    if slot == _TEST_SLOT:
        return TEST
    if slot == _VALIDATION_SLOT:
        return VALIDATION
    return TRAIN


def _split_by_district(frame: pd.DataFrame, seed: int) -> dict[str, str]:
    districts = _districts_by_favela_rate(frame)
    slots = _shuffled_slots(len(districts), seed)
    return {str(district): _split_of_slot(int(slot)) for district, slot in zip(districts, slots, strict=True)}


def assign_split(frame: pd.DataFrame, seed: int) -> pd.Series:
    return frame[GROUP].map(_split_by_district(frame, seed)).rename(SPLIT)


def rows_of(frame: pd.DataFrame, split: str) -> pd.DataFrame:
    return frame.loc[frame[SPLIT].eq(split)]


def _describe_split(rows: pd.DataFrame) -> dict[str, object]:
    return {
        "sectors": len(rows),
        "districts": rows[GROUP].nunique(),
        "favela_rate": round(float(rows[TARGET].mean()), 4),
    }


def describe_splits(frame: pd.DataFrame) -> pd.DataFrame:
    summary = {name: _describe_split(rows_of(frame, name)) for name in SPLITS}
    return pd.DataFrame.from_dict(summary, orient="index")


usable = usable.assign(**{SPLIT: assign_split(usable, SEED)})
describe_splits(usable)

,sectors,districts,favela_rate
train,21385,76,0.1455
validation,2786,10,0.1504
test,2508,10,0.1304


# Normalização

Os parâmetros são calculados só com o treino e aplicados igualmente aos três conjuntos, para que nenhuma
estatística de validação ou teste vaze para o modelo. Para cada feature contínua:

1. preenche vazios com a mediana do treino (robusta às caudas longas);
2. aplica `log1p` nas features positivas, sem teto e com cauda longa. Na tabela abaixo, a assimetria
   dessas quatro vai de 3 a 28 para no máximo 3,2 (`razao_sexo`, que segue com alguns setores extremos).
   `populacao` e `moradores_por_domicilio` ficariam mais assimétricas com log, então seguem lineares;
3. padroniza (z-score) com média e desvio do treino. As proporções também passam por isso, para todas as
   features ficarem na mesma escala (densidades na casa de 10⁴, proporções abaixo de 1).

As flags binárias ficam 0/1.

In [9]:
def skewness(frame: pd.DataFrame, names: list[str]) -> pd.DataFrame:
    values = frame[names]
    return pd.DataFrame({"raw": values.skew(), "log1p": values.transform(np.log1p).skew()}).round(2)


skewness(rows_of(usable, TRAIN), [POPULATION, "moradores_por_domicilio", *sorted(LOG_TRANSFORMED)])

,raw,log1p
populacao,0.87,-1.14
moradores_por_domicilio,-0.74,-1.16
densidade_domicilios_km2,7.13,-1.07
densidade_hab_km2,8.29,-1.40
razao_sexo,28.39,3.24
renda_media_responsavel,3.10,0.64


Cada feature guarda `fill_value` (mediana bruta), `log`, `mean` e `std` (já na escala do log, quando há
log). Esses parâmetros vão para o JSON, e a mesma transformação vale para dados novos:
`(log1p?(x.fillna(fill_value)) − mean) / std`.

In [10]:
_DECIMALS: Final = 6


@dataclass(frozen=True)
class FeatureScaling:
    fill_value: float
    log: bool
    mean: float
    std: float


Scaling = dict[str, FeatureScaling]


def _impute(values: pd.Series, fill_value: float) -> pd.Series:
    return values.fillna(fill_value)


def _transform(values: pd.Series, log: bool) -> pd.Series:
    return np.log1p(values) if log else values


def _nonzero_std(values: pd.Series) -> float:
    std = float(values.std(ddof=0))
    return std if std > 0 else 1.0


def _fit_feature(values: pd.Series, log: bool) -> FeatureScaling:
    fill_value = float(values.median())
    transformed = _transform(_impute(values, fill_value), log)
    return FeatureScaling(
        fill_value=fill_value,
        log=log,
        mean=float(transformed.mean()),
        std=_nonzero_std(transformed),
    )


def _scale_feature(values: pd.Series, scaling: FeatureScaling) -> pd.Series:
    transformed = _transform(_impute(values, scaling.fill_value), scaling.log)
    return ((transformed - scaling.mean) / scaling.std).round(_DECIMALS)


def fit_scaling(train: pd.DataFrame) -> Scaling:
    return {name: _fit_feature(train[name], name in LOG_TRANSFORMED) for name in CONTINUOUS}


def apply_scaling(frame: pd.DataFrame, scaling: Scaling) -> pd.DataFrame:
    scaled = {name: _scale_feature(frame[name], params) for name, params in scaling.items()}
    return frame.assign(**scaled)


def scaling_to_dict(scaling: Scaling) -> dict[str, dict[str, float | bool]]:
    return {name: asdict(params) for name, params in scaling.items()}


scaling = fit_scaling(rows_of(usable, TRAIN))
pd.DataFrame(scaling_to_dict(scaling)).T

,fill_value,log,mean,std
populacao,398.0,False,423.301567,216.933222
moradores_por_domicilio,2.7,False,2.62058,0.398849
densidade_hab_km2,18898.7097,True,9.826887,1.054637
densidade_domicilios_km2,8174.4324,True,9.028496,1.073008
pct_0a14,0.1611,False,0.160266,0.059782
pct_15a29,0.2079,False,0.208907,0.054333
pct_60mais,0.1806,False,0.184485,0.086576
razao_sexo,0.88515,True,0.635752,0.062702
pct_branca,0.5512,False,0.565791,0.189386
pct_preta,0.0931,False,0.094174,0.058962


Com a escala aplicada, o treino fica com média 0 e desvio 1 em todas as features contínuas. Valores extremos
não são cortados: proporções quase constantes geram z alto em poucos setores (`pct_banheiro_exclusivo = 0`
vira z ≈ −39). É sinal real e raro. Se o modelo for sensível a isso (rede neural), corte em ±5 na etapa de
treino.

In [11]:
def _model_columns() -> list[str]:
    return [*IDENTIFICATION, SPLIT, *LABEL, *FEATURES]


def _validate_features(frame: pd.DataFrame) -> None:
    missing = frame[list(FEATURES)].isna().sum()
    if missing.any():
        raise ValueError(f"features with missing values: {missing[missing > 0].to_dict()}")


def build_model_dataset(frame: pd.DataFrame, scaling: Scaling) -> pd.DataFrame:
    dataset = apply_scaling(frame, scaling)[_model_columns()]
    _validate_features(dataset)
    return dataset


dataset = build_model_dataset(usable, scaling)
rows_of(dataset, TRAIN)[list(CONTINUOUS)].agg(["mean", "std", "min", "max"]).T.round(2)

,mean,std,min,max
populacao,-0.0,1.0,-1.91,8.29
moradores_por_domicilio,-0.0,1.0,-4.06,4.46
densidade_hab_km2,-0.0,1.0,-8.44,3.92
densidade_domicilios_km2,-0.0,1.0,-7.53,3.87
pct_0a14,0.0,1.0,-2.68,5.68
pct_15a29,-0.0,1.0,-3.84,11.82
pct_60mais,-0.0,1.0,-2.13,6.31
razao_sexo,-0.0,1.0,-10.14,32.23
pct_branca,-0.0,1.0,-2.99,2.29
pct_preta,0.0,1.0,-1.60,7.16


# Salvar

| arquivo | conteúdo |
|---|---|
| `setores_sao_paulo_2022_tratado.csv` | base do modelo: identificação, `split`, rótulo e features normalizadas |
| `setores_sao_paulo_2022_excluidos.csv` | setores fora do modelo, com `exclusion_reason` e as features brutas |
| `setores_sao_paulo_2022_tratamento.json` | `seed`, contagens de exclusão, distritos de cada conjunto e parâmetros da normalização |

Com a mesma entrada e a mesma `SEED`, os arquivos saem idênticos byte a byte.

In [12]:
def _excluded_columns() -> list[str]:
    return [*IDENTIFICATION, EXCLUSION_REASON, *LABEL, *CONTINUOUS]


def _district_names(rows: pd.DataFrame) -> list[str]:
    return sorted(rows[GROUP].unique().tolist())


def _split_report(dataset: pd.DataFrame) -> dict[str, dict[str, object]]:
    return {
        name: {**_describe_split(rows_of(dataset, name)), "district_names": _district_names(rows_of(dataset, name))}
        for name in SPLITS
    }


def _exclusion_report(excluded: pd.DataFrame) -> dict[str, int]:
    counts = excluded[EXCLUSION_REASON].value_counts()
    return {str(reason): int(count) for reason, count in counts.items()}


def build_report(dataset: pd.DataFrame, excluded: pd.DataFrame, scaling: Scaling) -> dict[str, object]:
    return {
        "source": SOURCE.name,
        "key": KEY,
        "target": TARGET,
        "seed": SEED,
        "stratum_size": STRATUM_SIZE,
        "excluded": _exclusion_report(excluded),
        "splits": _split_report(dataset),
        "identification": list(IDENTIFICATION),
        "label": list(LABEL),
        "binary_features": list(BINARY),
        "scaled_features": scaling_to_dict(scaling),
    }


def save_outputs(dataset: pd.DataFrame, excluded: pd.DataFrame, report: dict[str, object]) -> None:
    dataset.to_csv(DATASET_PATH, index=False)
    excluded[_excluded_columns()].to_csv(EXCLUDED_PATH, index=False)
    REPORT_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")


save_outputs(dataset, partition.excluded, build_report(dataset, partition.excluded, scaling))
for path in (DATASET_PATH, EXCLUDED_PATH, REPORT_PATH):
    print("saved:", path.relative_to(ROOT))

saved: data/output/setores_sao_paulo_2022_tratado.csv
saved: data/output/setores_sao_paulo_2022_excluidos.csv
saved: data/output/setores_sao_paulo_2022_tratamento.json


# Conferência

Relê os arquivos salvos e confere: todo setor está em exatamente um dos dois arquivos, a chave segue com 15
dígitos, nenhum distrito aparece em dois conjuntos, as features não têm vazios e os parâmetros do JSON
reproduzem os valores normalizados a partir do CSV bruto.

In [13]:
saved_dataset = pd.read_csv(DATASET_PATH, dtype=_TEXT_COLUMNS)
saved_excluded = pd.read_csv(EXCLUDED_PATH, dtype=_TEXT_COLUMNS)
saved_scaling = {
    name: FeatureScaling(**params)
    for name, params in json.loads(REPORT_PATH.read_text(encoding="utf-8"))["scaled_features"].items()
}
raw_usable = sectors.set_index(KEY).loc[saved_dataset[KEY]].reset_index()

assert len(saved_dataset) + len(saved_excluded) == len(sectors)
assert set(saved_dataset[KEY]).isdisjoint(saved_excluded[KEY])
assert saved_dataset[KEY].str.fullmatch(_KEY_PATTERN).all()
assert saved_dataset.groupby(GROUP)[SPLIT].nunique().max() == 1
assert saved_dataset[list(FEATURES)].notna().all().all()
assert np.allclose(
    apply_scaling(raw_usable, saved_scaling)[list(CONTINUOUS)],
    saved_dataset[list(CONTINUOUS)],
    atol=1e-6,
)
print("ok")

ok
